# Step 0B — USCG PSIX XML Investigation

## Objective

Investigate the USCG PSIX / CGMIX XML source before designing ingestion
or canonical transformations.

Goals:

- identify the XML service endpoint and available operations
- retrieve a small representative response
- inspect the raw XML structure
- determine the grain of returned records
- identify vessel identifiers
- identify inspection, activity, and deficiency information
- document nested/repeating structures
- compare important fields with the Merchant Vessels source
- record observed data types and null/missing-value behavior
- identify candidate fields for the proposed canonical vessel model

## 1. Source Access

**Source:** U.S. Coast Guard CGMIX / PSIX  
**Format:** XML web service  
**Candidate endpoint:** `https://cgmix.uscg.mil/xml/PSIXData.asmx`

Unlike the Merchant Vessels source, PSIX is accessed through a web service
rather than a downloadable fixed-width snapshot.

The first goal is to inspect the available service operations and identify
a request that returns a small representative vessel sample.

In [1]:
import requests

PSIX_URL = "https://cgmix.uscg.mil/xml/PSIXData.asmx"

response = requests.get(
    PSIX_URL,
    timeout=30
)

print("Status:", response.status_code)
print("Content-Type:", response.headers.get("content-type"))
print("Response length:", len(response.content))

Status: 200
Content-Type: text/html; charset=utf-8
Response length: 17877


In [2]:
print(response.text[:2000])



<html>

    <head><link rel="alternate" type="text/xml" href="/xml/PSIXData.asmx?disco" />

    <style type="text/css">
    
		BODY { color: #000000; background-color: white; font-family: Verdana; margin-left: 0px; margin-top: 0px; }
		#content { margin-left: 30px; font-size: .70em; padding-bottom: 2em; }
		A:link { color: #336699; font-weight: bold; text-decoration: underline; }
		A:visited { color: #6699cc; font-weight: bold; text-decoration: underline; }
		A:active { color: #336699; font-weight: bold; text-decoration: underline; }
		A:hover { color: cc3300; font-weight: bold; text-decoration: underline; }
		P { color: #000000; margin-top: 0px; margin-bottom: 12px; font-family: Verdana; }
		pre { background-color: #e5e5cc; padding: 5px; font-family: Courier New; font-size: x-small; margin-top: -5px; border: 1px #f0f0e0 solid; }
		td { color: #000000; font-family: Verdana; font-size: .7em; }
		h2 { font-size: 1.5em; font-weight: bold; margin-top: 25px; margin-bottom: 10px; border-to

In [3]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(response.text, "html.parser")

operations = []

for a in soup.find_all("a", href=True):
    href = a["href"]
    text = a.get_text(strip=True)

    if href and text:
        operations.append((text, href))
 
operations[:30]

[('Service Description', 'PSIXData.asmx?WSDL'),
 ('getOperationControls', 'PSIXData.asmx?op=getOperationControls'),
 ('getOperationControlsXMLString',
  'PSIXData.asmx?op=getOperationControlsXMLString'),
 ('getVesselCases', 'PSIXData.asmx?op=getVesselCases'),
 ('getVesselCasesXMLString', 'PSIXData.asmx?op=getVesselCasesXMLString'),
 ('getVesselDeficiencies', 'PSIXData.asmx?op=getVesselDeficiencies'),
 ('getVesselDeficienciesXMLString',
  'PSIXData.asmx?op=getVesselDeficienciesXMLString'),
 ('getVesselDimensions', 'PSIXData.asmx?op=getVesselDimensions'),
 ('getVesselDimensionsXMLString',
  'PSIXData.asmx?op=getVesselDimensionsXMLString'),
 ('getVesselDocuments', 'PSIXData.asmx?op=getVesselDocuments'),
 ('getVesselDocumentsXMLString',
  'PSIXData.asmx?op=getVesselDocumentsXMLString'),
 ('getVesselInspectionTypes', 'PSIXData.asmx?op=getVesselInspectionTypes'),
 ('getVesselInspectionTypesXMLString',
  'PSIXData.asmx?op=getVesselInspectionTypesXMLString'),
 ('getVesselParticulars', 'PSIXDat

In [4]:
for name, href in operations:
    print(f"{name:40} {href}")

Service Description                      PSIXData.asmx?WSDL
getOperationControls                     PSIXData.asmx?op=getOperationControls
getOperationControlsXMLString            PSIXData.asmx?op=getOperationControlsXMLString
getVesselCases                           PSIXData.asmx?op=getVesselCases
getVesselCasesXMLString                  PSIXData.asmx?op=getVesselCasesXMLString
getVesselDeficiencies                    PSIXData.asmx?op=getVesselDeficiencies
getVesselDeficienciesXMLString           PSIXData.asmx?op=getVesselDeficienciesXMLString
getVesselDimensions                      PSIXData.asmx?op=getVesselDimensions
getVesselDimensionsXMLString             PSIXData.asmx?op=getVesselDimensionsXMLString
getVesselDocuments                       PSIXData.asmx?op=getVesselDocuments
getVesselDocumentsXMLString              PSIXData.asmx?op=getVesselDocumentsXMLString
getVesselInspectionTypes                 PSIXData.asmx?op=getVesselInspectionTypes
getVesselInspectionTypesXMLString     

In [5]:
# inspect method definition
method_url = (
    "https://cgmix.uscg.mil/xml/"
    "PSIXData.asmx?op=getVesselParticularsXMLString"
)

method_response = requests.get(method_url, timeout=30)

print("Status:", method_response.status_code)
print(method_response.text[:4000])

Status: 200


<html>

    <head><link rel="alternate" type="text/xml" href="/xml/PSIXData.asmx?disco" />

    <style type="text/css">
    
		BODY { color: #000000; background-color: white; font-family: Verdana; margin-left: 0px; margin-top: 0px; }
		#content { margin-left: 30px; font-size: .70em; padding-bottom: 2em; }
		A:link { color: #336699; font-weight: bold; text-decoration: underline; }
		A:visited { color: #6699cc; font-weight: bold; text-decoration: underline; }
		A:active { color: #336699; font-weight: bold; text-decoration: underline; }
		A:hover { color: cc3300; font-weight: bold; text-decoration: underline; }
		P { color: #000000; margin-top: 0px; margin-bottom: 12px; font-family: Verdana; }
		pre { background-color: #e5e5cc; padding: 5px; font-family: Courier New; font-size: x-small; margin-top: -5px; border: 1px #f0f0e0 solid; }
		td { color: #000000; font-family: Verdana; font-size: .7em; }
		h2 { font-size: 1.5em; font-weight: bold; margin-top: 25px; margin-bottom: 10p

In [6]:
method_soup = BeautifulSoup(method_response.text, "html.parser")

for tag in method_soup.find_all(["h1", "h2", "h3", "p", "pre", "code", "input"]):
    text = tag.get_text(" ", strip=True)
    if text:
        print(text) 

PSIXData
Click here for a complete list of operations.
getVesselParticularsXMLString
Returns XML String: VesselName(String): Vessel Name VesselCallSign(String): Vessel Call Sign CountryLookupName(String): Vessel Flag ServiceType(String): Vessel Service Type ServiceSubType(String): Vessel Service Sub Type CargoAuthorizationDescription(String): Vessel Cargo Authority Description ConstructionCompletedYear(String): Vessel Build Year StatusLookupName(String): Vessel Service Status OutOfServiceDate(DateTime): Out of Service Date - If Applicable VIN(String): Vessel Identification Number IdentificationTypeLookupName(String): Identification Type
Test
SOAP 1.1
The following is a sample SOAP 1.1 request and response.  The placeholders shown need to be replaced with actual values.
POST /xml/PSIXData.asmx HTTP/1.1
Host: cgmix.uscg.mil
Content-Type: text/xml; charset=utf-8
Content-Length: length SOAPAction: "https://cgmix.uscg.mil/getVesselParticularsXMLString"

<?xml version="1.0" encoding="utf-8"?

In [7]:
# get vessel particulars
import requests

soap_url = "https://cgmix.uscg.mil/xml/PSIXData.asmx"

vessel_id = 328660

soap_body = f"""<?xml version="1.0" encoding="utf-8"?>
<soap:Envelope
    xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance"
    xmlns:xsd="http://www.w3.org/2001/XMLSchema"
    xmlns:soap="http://schemas.xmlsoap.org/soap/envelope/">
  <soap:Body>
    <getVesselParticularsXMLString xmlns="https://cgmix.uscg.mil">
      <VesselID>{vessel_id}</VesselID>
    </getVesselParticularsXMLString>
  </soap:Body>
</soap:Envelope>
"""

headers = {
    "Content-Type": "text/xml; charset=utf-8",
    "SOAPAction": '"https://cgmix.uscg.mil/getVesselParticularsXMLString"',
}

response = requests.post(
    soap_url,
    data=soap_body,
    headers=headers,
    timeout=30,
)

print("Status:", response.status_code)
print("Content-Type:", response.headers.get("content-type"))
print(response.text[:3000])

Status: 503
Content-Type: text/html; charset=us-ascii
<!DOCTYPE HTML PUBLIC "-//W3C//DTD HTML 4.01//EN""http://www.w3.org/TR/html4/strict.dtd">
<HTML><HEAD><TITLE>Service Unavailable</TITLE>
<META HTTP-EQUIV="Content-Type" Content="text/html; charset=us-ascii"></HEAD>
<BODY><h2>Service Unavailable</h2>
<hr><p>HTTP Error 503. The service is unavailable.</p>
</BODY></HTML>



In [8]:
import time

for attempt in range(1, 4):
    response = requests.post(
        soap_url,
        data=soap_body,
        headers=headers,
        timeout=30,
    )

    print(f"Attempt {attempt}: {response.status_code}")

    if response.status_code == 200:
        break

    time.sleep(5)

Attempt 1: 200


In [9]:
print("Content-Type:", response.headers.get("content-type"))
print("Response length:", len(response.content))
print(response.text[:3000])

Content-Type: text/xml; charset=utf-8
Response length: 1196
<?xml version="1.0" encoding="utf-8"?><soap:Envelope xmlns:soap="http://schemas.xmlsoap.org/soap/envelope/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xmlns:xsd="http://www.w3.org/2001/XMLSchema"><soap:Body><getVesselParticularsXMLStringResponse xmlns="https://cgmix.uscg.mil"><getVesselParticularsXMLStringResult>&lt;NewDataSet&gt;
  &lt;VesselParticulars&gt;
    &lt;VesselName&gt;$5 COVER&lt;/VesselName&gt;
    &lt;ServiceType&gt;Recreational&lt;/ServiceType&gt;
    &lt;ServiceSubType&gt;General&lt;/ServiceSubType&gt;
    &lt;StatusLookupName&gt;Active&lt;/StatusLookupName&gt;
    &lt;ConstructionCompletedYear&gt;1989&lt;/ConstructionCompletedYear&gt;
    &lt;Identification&gt;947092&lt;/Identification&gt;
    &lt;IdentificationTypeLookupId&gt;6&lt;/IdentificationTypeLookupId&gt;
    &lt;IdentificationTypeLookupName&gt;Official Number (U.S.)&lt;/IdentificationTypeLookupName&gt;
    &lt;CountryLookupName&gt;UNITED ST

In [10]:
from lxml import etree

root = etree.fromstring(response.content)

namespaces = {
    "soap": "http://schemas.xmlsoap.org/soap/envelope/",
    "psix": "https://cgmix.uscg.mil",
}

result = root.find(
    ".//psix:getVesselParticularsXMLStringResult",
    namespaces,
)

print(result.text[:2000] if result is not None and result.text else "No result text")

<NewDataSet>
  <VesselParticulars>
    <VesselName>$5 COVER</VesselName>
    <ServiceType>Recreational</ServiceType>
    <ServiceSubType>General</ServiceSubType>
    <StatusLookupName>Active</StatusLookupName>
    <ConstructionCompletedYear>1989</ConstructionCompletedYear>
    <Identification>947092</Identification>
    <IdentificationTypeLookupId>6</IdentificationTypeLookupId>
    <IdentificationTypeLookupName>Official Number (U.S.)</IdentificationTypeLookupName>
    <CountryLookupName>UNITED STATES</CountryLookupName>
    <CountryLookupId>176948</CountryLookupId>
  </VesselParticulars>
</NewDataSet>


In [11]:
from lxml import etree

inner_xml = result.text
inner_root = etree.fromstring(inner_xml.encode("utf-8"))

record = inner_root.find(".//VesselParticulars")

psix_record = {
    child.tag: child.text
    for child in record
}

psix_record

{'VesselName': '$5 COVER',
 'ServiceType': 'Recreational',
 'ServiceSubType': 'General',
 'StatusLookupName': 'Active',
 'ConstructionCompletedYear': '1989',
 'Identification': '947092',
 'IdentificationTypeLookupId': '6',
 'IdentificationTypeLookupName': 'Official Number (U.S.)',
 'CountryLookupName': 'UNITED STATES',
 'CountryLookupId': '176948'}

In [12]:
import pandas as pd

df_psix_sample = pd.DataFrame([psix_record])

df_psix_sample

,VesselName,ServiceType,ServiceSubType,StatusLookupName,ConstructionCompletedYear,Identification,IdentificationTypeLookupId,IdentificationTypeLookupName,CountryLookupName,CountryLookupId
0,$5 COVER,Recreational,General,Active,1989,947092,6,Official Number (U.S.),UNITED STATES,176948


In [13]:
df_psix_sample.dtypes

VesselName                      str
ServiceType                     str
ServiceSubType                  str
StatusLookupName                str
ConstructionCompletedYear       str
Identification                  str
IdentificationTypeLookupId      str
IdentificationTypeLookupName    str
CountryLookupName               str
CountryLookupId                 str
dtype: object

Merchant Vessel Table Mapping

In [14]:
comparison = pd.DataFrame([
    {
        "psix_field": "VesselName",
        "merchant_field": "VesselName",
        "proposed_canonical_field": "vessel_name",
    },
    {
        "psix_field": "ServiceType",
        "merchant_field": "Service",
        "proposed_canonical_field": "vessel_service",
    },
    {
        "psix_field": "ConstructionCompletedYear",
        "merchant_field": "Build Year",
        "proposed_canonical_field": "build_year",
    },
    {
        "psix_field": "Identification",
        "merchant_field": "Official_Number",
        "proposed_canonical_field": "official_number",
    },
    {
        "psix_field": "CountryLookupName",
        "merchant_field": "Flag",
        "proposed_canonical_field": "flag_country",
    },
])

comparison

,psix_field,merchant_field,proposed_canonical_field
0,VesselName,VesselName,vessel_name
1,ServiceType,Service,vessel_service
2,ConstructionCompletedYear,Build Year,build_year
3,Identification,Official_Number,official_number
4,CountryLookupName,Flag,flag_country


In [15]:
psix_field_comparison = pd.DataFrame([
    {
        "psix_field": "VesselName",
        "merchant_field": "VesselName",
        "relationship": "direct overlap",
        "proposed_canonical_field": "vessel_name",
    },
    {
        "psix_field": "ServiceType",
        "merchant_field": "Service",
        "relationship": "similar concept",
        "proposed_canonical_field": "vessel_service",
    },
    {
        "psix_field": "ServiceSubType",
        "merchant_field": None,
        "relationship": "PSIX only",
        "proposed_canonical_field": "vessel_service_subtype",
    },
    {
        "psix_field": "StatusLookupName",
        "merchant_field": None,
        "relationship": "PSIX only",
        "proposed_canonical_field": "vessel_status",
    },
    {
        "psix_field": "ConstructionCompletedYear",
        "merchant_field": "Build Year",
        "relationship": "direct overlap",
        "proposed_canonical_field": "build_year",
    },
    {
        "psix_field": "Identification",
        "merchant_field": "Official_Number",
        "relationship": "conditional overlap",
        "proposed_canonical_field": "official_number",
    },
    {
        "psix_field": "IdentificationTypeLookupId",
        "merchant_field": None,
        "relationship": "PSIX only",
        "proposed_canonical_field": None,
    },
    {
        "psix_field": "IdentificationTypeLookupName",
        "merchant_field": None,
        "relationship": "PSIX only metadata",
        "proposed_canonical_field": "identifier_type",
    },
    {
        "psix_field": "CountryLookupName",
        "merchant_field": "Flag",
        "relationship": "related concept",
        "proposed_canonical_field": "flag_country",
    },
    {
        "psix_field": "CountryLookupId",
        "merchant_field": None,
        "relationship": "PSIX only",
        "proposed_canonical_field": None,
    },
])

psix_field_comparison

,psix_field,merchant_field,relationship,proposed_canonical_field
0,VesselName,VesselName,direct overlap,vessel_name
1,ServiceType,Service,similar concept,vessel_service
2,ServiceSubType,NaN,PSIX only,vessel_service_subtype
3,StatusLookupName,NaN,PSIX only,vessel_status
4,ConstructionCompletedYear,Build Year,direct overlap,build_year
5,Identification,Official_Number,conditional overlap,official_number
6,IdentificationTypeLookupId,NaN,PSIX only,NaN
7,IdentificationTypeLookupName,NaN,PSIX only metadata,identifier_type
8,CountryLookupName,Flag,related concept,flag_country
9,CountryLookupId,NaN,PSIX only,NaN
